# Analyzing queries: query history, query profile and caching

Query history lists every query on your SQL warehouses with its status, duration, user and source; the query profile opens one run and shows where the time went, operator by operator, with rows, bytes, pruning and spill. Databricks SQL also reuses work through caches. The query result cache returns a stored result when the same query runs again on unchanged tables, and the disk cache keeps recently read files on local SSDs. Read the profile before you resize anything.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Analyzing queries: query history, query profile and caching](https://www.datapreplab.com/hub#topic/query-performance) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** create a table with several files, compare a pruning-friendly filter with an unfriendly one, and see both caches at work.

**Part A: data in five files.** Two million rows, one every two seconds from 1 September (about 46 days). Each INSERT writes its own files covering about 9 days, so the files have separate time ranges. A table this small written in one statement is a single file, and then nothing can be pruned. Run Part C soon after, before any automatic maintenance compacts the files.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.events_lab (event_id BIGINT, event_ts TIMESTAMP, user_id INT);

In [ ]:
%sql
INSERT INTO workspace.default.events_lab
SELECT id, timestampadd(SECOND, id * 2, TIMESTAMP'2026-09-01 00:00:00'), CAST(id % 1000 AS INT) FROM range(0, 400000);

In [ ]:
%sql
INSERT INTO workspace.default.events_lab
SELECT id, timestampadd(SECOND, id * 2, TIMESTAMP'2026-09-01 00:00:00'), CAST(id % 1000 AS INT) FROM range(400000, 800000);

In [ ]:
%sql
INSERT INTO workspace.default.events_lab
SELECT id, timestampadd(SECOND, id * 2, TIMESTAMP'2026-09-01 00:00:00'), CAST(id % 1000 AS INT) FROM range(800000, 1200000);

In [ ]:
%sql
INSERT INTO workspace.default.events_lab
SELECT id, timestampadd(SECOND, id * 2, TIMESTAMP'2026-09-01 00:00:00'), CAST(id % 1000 AS INT) FROM range(1200000, 1600000);

In [ ]:
%sql
INSERT INTO workspace.default.events_lab
SELECT id, timestampadd(SECOND, id * 2, TIMESTAMP'2026-09-01 00:00:00'), CAST(id % 1000 AS INT) FROM range(1600000, 2000000);

In [ ]:
%sql
DESCRIBE DETAIL workspace.default.events_lab;

**Part B: the plan.** Compare the plans of the two filters:

In [ ]:
%sql
EXPLAIN FORMATTED
SELECT count(*) FROM workspace.default.events_lab
WHERE event_ts >= TIMESTAMP'2026-10-15 00:00:00' AND event_ts < TIMESTAMP'2026-10-16 00:00:00';

In [ ]:
%sql
EXPLAIN FORMATTED
SELECT count(*) FROM workspace.default.events_lab
WHERE date_format(event_ts, 'yyyy-MM-dd') = '2026-10-15';

**Part C: profile both (SQL editor).** Predict which query reads fewer files, then run each query (without EXPLAIN) in the SQL editor. In **Query History**, open each run and note **Files read**, **Files pruned**, **Bytes read** and **Bytes read from cache**.

**Part D: the caches.** Run the range query again: it returns almost instantly from the **result cache**, and Query History shows "Query profile is unavailable for cached results". Then, for a fair timing, run:

```text
SET use_cached_result = false;
```

and run the query once more. It executes again and has a profile; look at **Bytes read from cache**, which is the **disk cache** at work.

Expected evidence:

```text
Part A -> DESCRIBE DETAIL shows numFiles of at least 5 (9 on Free Edition: an INSERT can write more than one file) ; sizeInBytes about 65 KB
Part B -> the range filter appears as a data filter on event_ts; the formatted filter wraps the column in date_format
Part C -> both count 43,200 rows ; range filter: files read 1, files pruned 8 (89% of files, 92% of bytes) ; date_format filter: files read 9, files pruned 0
Part D -> repeat run: no profile (result cache) ; with use_cached_result = false: a profile again, Bytes read from cache 100%, much faster than the first cold run
```

**Cleanup:** `DROP TABLE workspace.default.events_lab;`